# Lesson 2b Demo: The Difficulty of Testing

LLM outputs are **non-deterministic by design**. The same prompt produces different (but equally valid) answers every time. This notebook explores what we *can* test and where traditional assertions break down.

Run cells in order.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print("Client ready")


Client ready


## The Difficulty of Testing

In traditional software, `sort([3,1,2])` must return `[1,2,3]`. One correct answer, deterministic, easy to assert.

LLM outputs are **non-deterministic by design**. The same prompt produces different (but equally valid) answers every time. No `assert ==` can capture "correct" when correct is a spectrum, not a point.

We'll see this with three tasks — open-ended QnA, document-based QnA, and structured extraction — and observe how much the outputs vary.

### What *can* we test?

- **Structural checks** — is the output non-empty? Is it within length bounds?
- **Property checks** — does a "one sentence" answer actually contain one sentence?
- **LLM-as-judge** — ask another LLM to grade the output (we'll cover this in depth later)

### Example 1: Open-ended QnA

Same prompt, three runs. All answers are correct, none are identical.

In [2]:
prompt = "Explain Software 3.0 in one sentence."

# Run the SAME prompt three times
for i in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    answer = r.choices[0].message.content
    print(f"Run {i}: {answer}\n")

# All different, all correct. What would you assert?
assert answer is not None and len(answer) > 0, "Response should not be empty"
assert len(answer) < 500, "A one-sentence answer shouldn't be 500+ chars"
print("✓ Structural checks pass — but they don't tell us if the answer is *good*.")
print("  This is the gap we'll address with evaluation methods in later lessons.")

Run 1: Software 3.0 refers to a paradigm where software is primarily created by training machine learning models on data rather than being explicitly programmed with traditional code.

Run 2: Software 3.0 refers to the paradigm where software development is driven primarily by machine learning models that learn from data, rather than being explicitly programmed with traditional code.

Run 3: Software 3.0 refers to the paradigm where software is primarily created by training machine learning models on data, rather than by explicitly programming rules.

✓ Structural checks pass — but they don't tell us if the answer is *good*.
  This is the gap we'll address with evaluation methods in later lessons.


### Example 2: Document-based QnA

Now the model has a "right" answer grounded in a document — but it still paraphrases differently each time.

In [3]:
documents = {
    "returns": "Customers may return items within 30 days of purchase. Electronics must be unopened. Refunds are issued to the original payment method within 5 business days.",
    "shipping": "Standard shipping takes 5-7 business days. Express shipping (2-day) is available for $12.99. Free shipping on orders over $75.",
    "warranty": "All electronics come with a 2-year manufacturer warranty. Accidental damage is not covered. Extended warranty available for $49.99/year.",
}

context = "\n\n".join(f"[{name.upper()}]\n{text}" for name, text in documents.items())
question = "I bought a laptop 3 weeks ago. Can I still return it, and what warranty do I get?"

for i in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": f"Answer based only on these documents:\n\n{context}"},
            {"role": "user", "content": question},
        ],
        temperature=0.7,
    )
    print(f"Run {i}: {r.choices[0].message.content}\n")

Run 1: You can still return the laptop since it has been only 3 weeks (within the 30-day return window). However, the laptop must be unopened to qualify for a return. 

Regarding warranty, your laptop comes with a 2-year manufacturer warranty. Note that accidental damage is not covered under this warranty. You also have the option to purchase an extended warranty for $49.99 per year if you want additional coverage.

Run 2: Yes, you can still return the laptop since it has been only 3 weeks (within the 30-day return window). However, the laptop must be unopened to be eligible for return.

Regarding warranty, your laptop comes with a 2-year manufacturer warranty. Note that this warranty does not cover accidental damage. You also have the option to purchase an extended warranty for $49.99 per year if you wish.

Run 3: Since you bought the laptop 3 weeks ago, you can still return it within the 30-day return window. However, the laptop must be unopened to be eligible for return. Regarding w

### Example 3: Named Entity Recognition (structured output)

Structured tasks like NER are more constrained — the model must return specific entities from the text. There's less room for variation, but it still happens: entity boundaries, classifications, and formatting can differ across runs.

In [4]:
import json

ner_prompt = """Extract all named entities from the text below. Return valid JSON only.

Format: {"entities": [{"text": "...", "type": "PERSON|ORG|LOCATION|DATE"}]}

Text: "On March 5th, EU commissioner Teresa Ribera met with OpenAI's Sam Altman in Brussels to discuss the AI Act's impact on European startups."
"""

for i in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": ner_prompt}],
        temperature=0.7,
    )
    raw = r.choices[0].message.content
    print(f"--- Run {i} ---")
    try:
        parsed = json.loads(raw)
        for e in parsed["entities"]:
            print(f"  {e['type']:10s} {e['text']}")
    except (json.JSONDecodeError, KeyError):
        print(f"  (parse error) {raw[:200]}")
    print()

print("Notice: the core entities are stable, but details like exact spans,")
print("classifications (ORG vs MISC), and whether 'EU' is separate may vary.")

--- Run 1 ---
  (parse error) ```json
{"entities": [{"text": "March 5th", "type": "DATE"}, {"text": "EU", "type": "ORG"}, {"text": "Teresa Ribera", "type": "PERSON"}, {"text": "OpenAI", "type": "ORG"}, {"text": "Sam Altman", "type

--- Run 2 ---
  (parse error) ```json
{"entities": [{"text": "March 5th", "type": "DATE"}, {"text": "EU", "type": "ORG"}, {"text": "Teresa Ribera", "type": "PERSON"}, {"text": "OpenAI", "type": "ORG"}, {"text": "Sam Altman", "type

--- Run 3 ---
  (parse error) ```json
{"entities": [{"text": "March 5th", "type": "DATE"}, {"text": "EU", "type": "ORG"}, {"text": "Teresa Ribera", "type": "PERSON"}, {"text": "OpenAI", "type": "ORG"}, {"text": "Sam Altman", "type

Notice: the core entities are stable, but details like exact spans,
classifications (ORG vs MISC), and whether 'EU' is separate may vary.


## Exercise: Extract plot and characters from a mystery story

The file `prompting/sherlock-short.txt` contains the beginning of *A Scandal in Bohemia* from *The Adventures of Sherlock Holmes*.

**Your task:** Write a prompt that extracts:
1. A **plot summary** (what happens in this passage)
2. The **names of all characters** mentioned

Try it below. Then think about: *how would you know if the output is good?* We'll tackle that question systematically in the next notebook.

In [ ]:
from pathlib import Path

text_path = Path("labs/02_standalone_agents/prompting/sherlock-short.txt")
if not text_path.exists():
    text_path = Path("prompting/sherlock-short.txt")
story = text_path.read_text()
print(f"Loaded {len(story):,} characters\n")

# TODO: Write your extraction prompt here
extraction_prompt = """
YOUR PROMPT HERE
"""

r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": f"{extraction_prompt}\n\n{story}"},
    ],
    temperature=0.7,
)
print(r.choices[0].message.content)

## Exercise: Jinja2 Prompt Templates

In production systems, prompts are rarely hardcoded strings. Instead, teams use **template engines** like [Jinja2](https://jinja.palletsprojects.com/) to separate prompt *structure* from prompt *data*.

This makes prompts:
- **Reusable** — swap in different texts, formats, or constraints without rewriting
- **Testable** — render the template with known inputs and inspect the result
- **Maintainable** — change the template once, all callers benefit

**Your task:** Complete the Jinja2 template below to extract the same plot summary and character list from the Sherlock Holmes text. The template receives two variables:
- `story_text` — the passage to analyse
- `output_format` — either `"json"` or `"markdown"`

Fill in the template so the model returns structured output in the requested format.

In [ ]:
from jinja2 import Template
from pathlib import Path

# Load the story text
text_path = Path("labs/02_standalone_agents/prompting/sherlock-short.txt")
if not text_path.exists():
    text_path = Path("prompting/sherlock-short.txt")
story = text_path.read_text()

# Load the Jinja2 prompt template from file
# TODO: Edit prompting/plot_prompt_template.j2 to complete the template.
#   Variables available: {{ story_text }}, {{ output_format }}
#   Use {% if %} / {% else %} / {% endif %} to handle the two output formats.
template_path = Path("labs/02_standalone_agents/prompting/plot_prompt_template.j2")
if not template_path.exists():
    template_path = Path("prompting/plot_prompt_template.j2")
prompt_template = Template(template_path.read_text())

# Render the template with our variables
rendered_prompt = prompt_template.render(
    story_text=story,
    output_format="json",  # try changing to "markdown" after it works
)

# Inspect what the template produced (useful for debugging!)
print("=== Rendered prompt (first 500 chars) ===")
print(rendered_prompt[:500])
print("...\n")

# Send to the model
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": rendered_prompt}],
    temperature=0.7,
)
print("=== Model output ===")
print(r.choices[0].message.content)